# Capstone: Support Agent with Redis Memory, Vector Recall, Async Queue, and Human Approval

**Course deliverable:** Extend Lab 2 into a small support agent with:

- **Redis short-term memory** + **vector-store long-term recall**
- **At least 3 tools**, including **one async queue-backed tool**
- **Routing + chaining** with a **human approval gate**
- **Tracing, retries, and prompt caching**

## Architecture at a glance

```text
User message
   │
   ▼
[Router / Orchestrator]
   ├─ FAQ / policy route ───────────────► search_kb() ─► vector store
   ├─ Order lookup route ───────────────► lookup_order() ─► SQLite
   ├─ Refund / cancel route ────────────► approval gate
   │                                        ├─ if not approved: pause + save pending action in Redis
   │                                        └─ if approved: create_refund_request() + send_followup_email()
   └─ Async follow-up route ────────────► send_followup_email() ─► Redis Stream queue ─► worker ─► check_job()


In [1]:
!apt-get update -qq
!apt-get install -y redis-server -qq
!pip install redis sentence-transformers scikit-learn pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../0-libjemalloc2_5.2.1-4ubuntu1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.2.1-4ubuntu1) ...
Selecting previously unselected package liblua5.1-0:amd64.
Preparing to unpack .../1-liblua5.1-0_5.1.5-8.1build4_amd64.deb ...
Unpacking liblua5.1-0:amd64 (5.1.5-8.1build4) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../2-liblzf1_3.6-3_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-3) ...
Selecting previously unselected package lua-bitop:amd64.
Preparing to unpack .../3-lua-bitop_1.0.2-5_amd64.deb ...
Unpacking lua-bitop:amd64 (1.0.2-5) ...
Selecting previously unselected package lua-cjson:amd64.
Preparing to unpack .../4-

In [2]:
import os
import re
import time
import json
import uuid
import math
import sqlite3
import hashlib
import threading
from datetime import datetime
from typing import Dict, List, Any, Optional

import pandas as pd

# Redis is used for:
# 1) short-term memory (session state)
# 2) async job queue + job status
try:
    import redis
except Exception:
    redis = None

# Embedding stack for the long-term recall vector store.
# If unavailable, we will use a tiny bag-of-words fallback.
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception:
    SentenceTransformer = None
    cosine_similarity = None


## 2) Redis short-term memory

Redis is used as the **short-term memory layer** for the support agent. It stores:
- the recent session history for a user / support conversation
- any **pending approval action** (for example, a refund request waiting for human approval)
- async **job status** for queued follow-up work



In [3]:

USE_REAL_REDIS = True

class InMemoryRedis:
    """
    Minimal Redis-like fallback used when a real Redis server is not available.
    It supports:
      - key/value get/set
      - Redis Streams-like queue operations used in this notebook
    """
    def __init__(self):
        self.kv = {}
        self.streams = {}
        self.groups = {}

    def ping(self):
        return True

    def get(self, key):
        value = self.kv.get(key)
        if isinstance(value, str):
            return value.encode()
        return value

    def set(self, key, value):
        self.kv[key] = value
        return True

    def xadd(self, stream_name, fields):
        self.streams.setdefault(stream_name, [])
        msg_id = f"{int(time.time() * 1000)}-{len(self.streams[stream_name])}"
        self.streams[stream_name].append((msg_id, fields))
        return msg_id

    def xgroup_create(self, stream_name, group_name, id='0', mkstream=True):
        self.groups.setdefault((stream_name, group_name), 0)

    def xreadgroup(self, groupname, consumername, streams, count=1, block=1000):
        """
        Very small mock implementation of a consumer-group read.
        """
        out = []
        for stream_name, _ in streams.items():
            items = self.streams.get(stream_name, [])
            idx = self.groups.get((stream_name, groupname), 0)
            if idx < len(items):
                batch = items[idx: idx + count]
                self.groups[(stream_name, groupname)] = idx + len(batch)
                out.append((
                    stream_name.encode(),
                    [
                        (
                            msg_id.encode(),
                            {k.encode(): str(v).encode() for k, v in payload.items()}
                        )
                        for msg_id, payload in batch
                    ]
                ))
        return out

    def xack(self, stream_name, group_name, msg_id):
        return 1


def get_redis_client():
    """
    Try to connect to a real local Redis server.
    If unavailable, return the in-memory fallback.
    """
    if USE_REAL_REDIS and redis is not None:
        try:
            client = redis.Redis(host="localhost", port=6379, decode_responses=False)
            client.ping()
            return client
        except Exception:
            pass
    return InMemoryRedis()


r = get_redis_client()
print("Redis connected:", bool(r.ping()))


Redis connected: True


## 3) SQLite support database
### Tables
- **`orders`**: order metadata used for order lookup and refund context
- **`refund_requests`**: refund actions created after approval


In [4]:

# Create an in-memory SQLite database for support operations.
conn = sqlite3.connect(":memory:", check_same_thread=False)
cur = conn.cursor()

cur.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_email TEXT,
    item TEXT,
    status TEXT,
    amount REAL,
    created_at TEXT
)
""")

cur.execute("""
CREATE TABLE refund_requests (
    request_id TEXT PRIMARY KEY,
    order_id TEXT,
    reason TEXT,
    status TEXT,
    created_at TEXT
)
""")

# Seed the database with a few example orders.
sample_orders = [
    ("1001", "siri@example.com", "Bluetooth Speaker", "shipped", 79.99, "2026-06-20"),
    ("1002", "siri@example.com", "Gaming Mouse", "processing", 49.99, "2026-06-22"),
    ("1003", "siri@example.com", "USB-C Dock", "delivered", 129.00, "2026-06-18"),
]

cur.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)", sample_orders)
conn.commit()

pd.read_sql_query("SELECT * FROM orders", conn)


,order_id,customer_email,item,status,amount,created_at
0,1001,siri@example.com,Bluetooth Speaker,shipped,79.99,2026-06-20
1,1002,siri@example.com,Gaming Mouse,processing,49.99,2026-06-22
2,1003,siri@example.com,USB-C Dock,delivered,129.00,2026-06-18


## 4) Long-term recall with a vector store

Here, long-term recall is implemented as a small **support knowledge base** that the agent can search semantically.

### Knowledge base examples
- return / refund policy
- cancellation policy
- shipping FAQ
- address change rules

### Implementation approach
- If `sentence-transformers` is available, we embed KB chunks with **`all-MiniLM-L6-v2`**
- Otherwise, we fall back to a tiny **bag-of-words** embedding

Either way, the `search_kb(query)` tool behaves like a long-term recall tool:
- it retrieves the most relevant policy / FAQ snippets
- it supports the agent’s FAQ and policy-answering routes


In [5]:

KB_DOCS = [
    {
        "id": "policy_returns",
        "text": "Damaged items are eligible for return or refund within 30 days of delivery. Photos may be requested for verification."
    },
    {
        "id": "policy_cancellations",
        "text": "Orders in processing can be cancelled immediately. Orders already shipped require manual review and may not be cancellable."
    },
    {
        "id": "policy_refunds",
        "text": "Approved refunds are typically returned to the original payment method within 5 to 7 business days."
    },
    {
        "id": "policy_shipping",
        "text": "Standard shipping updates appear after carrier scan. Delivered orders with missing items should be escalated to support."
    },
    {
        "id": "faq_address_change",
        "text": "Address changes after shipment are restricted and require human approval to avoid delivery errors."
    },
]


class SimpleVectorStore:
    """
    Minimal vector store abstraction for the support KB.

    - Preferred mode: SentenceTransformer embeddings
    - Fallback mode: bag-of-words normalized vectors

    This is enough to demonstrate the capstone's long-term recall requirement.
    """
    def __init__(self, docs):
        self.docs = docs
        self.model = None
        self.doc_vectors = None
        self.vocab = None
        self._build()

    def _bow_embed(self, texts):
        tokenized = []
        vocab = self.vocab or {}
        build_vocab = self.vocab is None

        for text in texts:
            tokens = re.findall(r"[a-zA-Z0-9']+", text.lower())
            tokenized.append(tokens)
            if build_vocab:
                for token in tokens:
                    if token not in vocab:
                        vocab[token] = len(vocab)

        if build_vocab:
            self.vocab = vocab

        vectors = []
        for tokens in tokenized:
            vec = [0.0] * len(self.vocab)
            for token in tokens:
                if token in self.vocab:
                    vec[self.vocab[token]] += 1.0

            norm = math.sqrt(sum(v * v for v in vec)) or 1.0
            vectors.append([v / norm for v in vec])

        return vectors

    def _build(self):
        texts = [doc["text"] for doc in self.docs]

        # Preferred path: dense embeddings
        if SentenceTransformer is not None:
            try:
                self.model = SentenceTransformer("all-MiniLM-L6-v2")
                self.doc_vectors = self.model.encode(texts)
                return
            except Exception:
                self.model = None

        # Fallback path: bag-of-words embeddings
        self.doc_vectors = self._bow_embed(texts)

    def search(self, query, top_k=3):
        """
        Return the top_k most similar KB chunks for the query.
        """
        if self.model is not None and cosine_similarity is not None:
            query_vector = self.model.encode([query])
            sims = cosine_similarity(query_vector, self.doc_vectors)[0]
        else:
            query_vector = self._bow_embed([query])[0]
            sims = [sum(a * b for a, b in zip(query_vector, doc_vec)) for doc_vec in self.doc_vectors]

        ranked = sorted(zip(self.docs, sims), key=lambda x: x[1], reverse=True)[:top_k]
        return [
            {"id": doc["id"], "text": doc["text"], "score": float(score)}
            for doc, score in ranked
        ]


vector_store = SimpleVectorStore(KB_DOCS)
vector_store.search("refund for damaged delivered item")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[{'id': 'policy_returns',
  'text': 'Damaged items are eligible for return or refund within 30 days of delivery. Photos may be requested for verification.',
  'score': 0.7333954572677612},
 {'id': 'policy_refunds',
  'text': 'Approved refunds are typically returned to the original payment method within 5 to 7 business days.',
  'score': 0.462956041097641},
 {'id': 'policy_shipping',
  'text': 'Standard shipping updates appear after carrier scan. Delivered orders with missing items should be escalated to support.',
  'score': 0.3489614725112915}]

## 5) Tracing, retries, and prompt caching


### 5.1 Tracing
Every major orchestration step or tool call writes a structured event to `TRACE_LOG`.
Examples:
- user message received
- tool called
- cache hit
- retry attempt
- worker completed async job

### 5.2 Prompt caching
Repeated KB queries are cached in `PROMPT_CACHE`, keyed by a normalized hash of the request.
This avoids recomputing the same retrieval result in the same run.

### 5.3 Retries
The helper `with_retry(...)` wraps operations that may fail transiently.
In a real system, this would be used for:
- external APIs
- Redis operations
- vector retrieval service calls
- email provider calls



In [6]:

TRACE_LOG = []
PROMPT_CACHE = {}


def now_iso():
    return datetime.utcnow().isoformat()


def trace(event_type, **kwargs):
    """
    Append a structured trace event to the in-memory trace log.
    """
    event = {"ts": now_iso(), "event": event_type, **kwargs}
    TRACE_LOG.append(event)
    return event


def cache_key(prefix, payload):
    """
    Build a stable cache key for prompt / retrieval caching.
    """
    raw = json.dumps(payload, sort_keys=True, default=str)
    return prefix + ":" + hashlib.sha256(raw.encode()).hexdigest()


def get_session_key(session_id):
    return f"support:session:{session_id}"


def load_session(session_id):
    """
    Load the short-term session state from Redis.

    Stored state includes:
    - conversation history
    - pending approval action (if any)
    """
    raw = r.get(get_session_key(session_id))
    if not raw:
        return {"session_id": session_id, "history": [], "pending": None}

    if isinstance(raw, bytes):
        raw = raw.decode()

    return json.loads(raw)


def save_session(session_id, state):
    r.set(get_session_key(session_id), json.dumps(state))
    return state


def append_history(session_id, role, content):
    """
    Save a conversation turn into short-term memory.
    """
    state = load_session(session_id)
    state["history"].append({
        "role": role,
        "content": content,
        "ts": now_iso()
    })
    save_session(session_id, state)
    return state


def with_retry(fn, *args, retries=3, base_delay=0.25, **kwargs):
    """
    Simple retry wrapper with exponential backoff.

    This demonstrates the capstone's reliability requirement.
    """
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            result = fn(*args, **kwargs)
            trace(
                "retry_success" if attempt > 1 else "call_success",
                fn=getattr(fn, "__name__", "anonymous"),
                attempt=attempt
            )
            return result
        except Exception as e:
            last_err = e
            trace(
                "retry_error",
                fn=getattr(fn, "__name__", "anonymous"),
                attempt=attempt,
                error=str(e)
            )
            if attempt < retries:
                time.sleep(base_delay * (2 ** (attempt - 1)))

    raise last_err


## 6) Tool implementations

### Tool 1 — `search_kb(query, top_k=3)`
Long-term recall over the support KB.

### Tool 2 — `lookup_order(order_id)`
Read operational order data from SQLite.

### Tool 3 — `create_refund_request(order_id, reason)`
Create a refund request record after approval.

### Tool 4 — `send_followup_email(to, subject, body)` **(async queue-backed tool)**
Push a job to a Redis Stream and return a `job_id` immediately.

### Tool 5 — `check_job(job_id)`
Read async job status from Redis.

In [7]:

def search_kb(query, top_k=3):
    """
    Long-term recall tool.

    Searches the support KB and returns the top_k most relevant chunks.
    Results are cached so repeated queries don't recompute retrieval.
    """
    key = cache_key("kb", {"query": query, "top_k": top_k})

    if key in PROMPT_CACHE:
        trace("cache_hit", tool="search_kb", query=query)
        return {"cached": True, "results": PROMPT_CACHE[key]}

    trace("tool_call", tool="search_kb", query=query)
    results = with_retry(vector_store.search, query, top_k=top_k)
    PROMPT_CACHE[key] = results
    return {"cached": False, "results": results}


def lookup_order(order_id):
    """
    Operational read tool.

    Returns order details from SQLite. Used by:
    - order lookup route
    - refund / cancellation routes
    """
    trace("tool_call", tool="lookup_order", order_id=order_id)

    row = cur.execute(
        "SELECT * FROM orders WHERE order_id = ?",
        (order_id,)
    ).fetchone()

    if not row:
        return {"found": False, "order_id": order_id}

    cols = ["order_id", "customer_email", "item", "status", "amount", "created_at"]
    return {"found": True, "order": dict(zip(cols, row))}


def create_refund_request(order_id, reason):
    """
    Sensitive write tool.

    This simulates creating a refund request in an operational system.
    In the orchestrator, it is only called after the approval gate passes.
    """
    trace("tool_call", tool="create_refund_request", order_id=order_id, reason=reason)

    request_id = "rr_" + uuid.uuid4().hex[:8]
    cur.execute(
        "INSERT INTO refund_requests VALUES (?, ?, ?, ?, ?)",
        (request_id, order_id, reason, "submitted", now_iso())
    )
    conn.commit()

    return {
        "request_id": request_id,
        "status": "submitted",
        "order_id": order_id
    }


# Redis Stream configuration for the async queue-backed tool.
STREAM_NAME = "support_jobs"
GROUP_NAME = "support_workers"
CONSUMER_NAME = "worker-1"


def ensure_consumer_group():
    """
    Ensure the Redis Stream consumer group exists.
    """
    try:
        r.xgroup_create(STREAM_NAME, GROUP_NAME, id="0", mkstream=True)
    except Exception:
        # It's fine if the group already exists.
        pass


ensure_consumer_group()


def send_followup_email(to, subject, body):
    """
    Async queue-backed tool.

    Instead of sending the email inline, the tool enqueues a job into a Redis Stream.
    A background worker will later consume the job and mark it completed.

    This is the capstone's required async queue-backed tool.
    """
    trace("tool_call", tool="send_followup_email", to=to, subject=subject)

    job_id = "job_" + uuid.uuid4().hex[:8]
    payload = {
        "job_id": job_id,
        "type": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "created_at": now_iso()
    }

    stream_id = r.xadd(STREAM_NAME, payload)

    # Save initial job status so the user can check it later.
    r.set(
        f"job:{job_id}",
        json.dumps({
            "job_id": job_id,
            "status": "queued",
            "stream_id": str(stream_id)
        })
    )

    return {"job_id": job_id, "status": "queued"}


def check_job(job_id):
    """
    Poll async job status from Redis.
    """
    raw = r.get(f"job:{job_id}")
    if not raw:
        return {"job_id": job_id, "status": "unknown"}

    if isinstance(raw, bytes):
        raw = raw.decode()

    return json.loads(raw)


## 7) Async worker for the queue-backed tool

The queue-backed tool above only **enqueues** a job.  
This section implements a background **worker** that consumes the Redis Stream and simulates sending the email.

This demonstrates a realistic async pattern:
1. user-facing tool returns immediately with a **job id**
2. background worker performs the slow / external work
3. user can call `check_job(job_id)` later


In [8]:

WORKER_STOP = False


def process_email_job(job):
    """
    Simulated slow external operation.
    In a real system, this would call an email provider API.
    """
    time.sleep(0.5)
    return {
        "delivered": True,
        "provider_message_id": uuid.uuid4().hex[:10]
    }


def worker_loop(max_idle_loops=20):
    """
    Background worker that reads from the Redis Stream and processes queued jobs.
    """
    idle = 0

    while not WORKER_STOP and idle < max_idle_loops:
        records = r.xreadgroup(
            groupname=GROUP_NAME,
            consumername=CONSUMER_NAME,
            streams={STREAM_NAME: '>'},
            count=1,
            block=500
        )

        if not records:
            idle += 1
            continue

        idle = 0

        for stream_name, messages in records:
            for msg_id, fields in messages:
                decoded = {}
                for k, v in fields.items():
                    key = k.decode() if isinstance(k, bytes) else k
                    val = v.decode() if isinstance(v, bytes) else v
                    decoded[key] = val

                job_id = decoded["job_id"]
                trace("worker_received", job_id=job_id, stream_id=str(msg_id))

                try:
                    result = with_retry(process_email_job, decoded, retries=2)

                    r.set(
                        f"job:{job_id}",
                        json.dumps({
                            "job_id": job_id,
                            "status": "completed",
                            "result": result,
                            "completed_at": now_iso()
                        })
                    )

                    try:
                        r.xack(STREAM_NAME, GROUP_NAME, msg_id)
                    except Exception:
                        pass

                    trace("worker_completed", job_id=job_id)

                except Exception as e:
                    r.set(
                        f"job:{job_id}",
                        json.dumps({
                            "job_id": job_id,
                            "status": "failed",
                            "error": str(e),
                            "failed_at": now_iso()
                        })
                    )
                    trace("worker_failed", job_id=job_id, error=str(e))


def run_worker_in_background():
    """
    Start the async worker in a daemon thread so demo scenarios can enqueue jobs.
    """
    thread = threading.Thread(
        target=worker_loop,
        kwargs={"max_idle_loops": 100},
        daemon=True
    )
    thread.start()
    return thread


## 8) Routing and human approval policy

### Intent routing
The orchestrator uses a simple intent classifier to map a user request to one of four routes:

1. **FAQ / policy** → `search_kb(...)`
2. **Order lookup** → `lookup_order(...)`
3. **Refund / cancellation** → `lookup_order(...)` + approval gate + action tool
4. **Async follow-up** → `send_followup_email(...)`

### Human approval gate
Certain actions are **sensitive** and must be explicitly approved:
- refunds
- cancellations
- address changes (illustrated in policy logic)

If approval is needed, the agent:
1. saves a **pending action** in Redis short-term memory
2. returns an **approval required** response
3. resumes the action only after explicit approval


In [9]:

def extract_order_id(text):
    """
    Pull a simple numeric order id like 1002 from the user message.
    """
    match = re.search(r'\b(10\d{2,})\b', text)
    return match.group(1) if match else None


def classify_intent(user_message):
    """
    Very small rule-based router for the capstone demo.
    """
    text = user_message.lower()

    if any(word in text for word in ["refund", "damaged", "broken"]):
        return "refund_request"
    if any(word in text for word in ["cancel", "cancellation"]):
        return "cancel_request"
    if any(word in text for word in ["where is", "status", "track", "order"]):
        return "order_lookup"
    if any(word in text for word in ["email", "notify", "follow up"]):
        return "async_followup"

    return "faq"


def needs_approval(intent, context):
    """
    Human approval policy.

    Sensitive actions must be explicitly approved before execution.
    """
    if intent in {"refund_request", "cancel_request", "address_change"}:
        return True

    order = context.get("order")
    if intent == "cancel_request" and order and order.get("status") == "shipped":
        return True

    return False


## 9) Orchestrator / support agent

This is the main **routing + chaining** layer.

### Route A — FAQ / policy
- call `search_kb(...)`
- answer from retrieved KB snippets

### Route B — Order lookup
- call `lookup_order(...)`
- optionally add a shipping KB note

### Route C — Refund / cancellation
- lookup order
- search KB for policy context
- if approval is needed:
  - save pending action in Redis
  - pause and ask for approval
- if approved:
  - create refund request
  - queue a follow-up email

### Route D — Async follow-up
- enqueue an email / follow-up job
- return the `job_id`


In [10]:

def support_agent(session_id, user_message, approved=False):
    """
    Main support orchestrator.

    Responsibilities:
    - save the incoming user turn to short-term memory
    - classify the intent / route
    - chain tools as needed
    - enforce human approval for sensitive actions
    - write assistant responses back to short-term memory
    """
    append_history(session_id, "user", user_message)
    trace("user_message", session_id=session_id, text=user_message)

    intent = classify_intent(user_message)
    order_id = extract_order_id(user_message)
    context = {"intent": intent, "order_id": order_id}

    # ------------------------------------------------------------
    # Route A: FAQ / policy question
    # ------------------------------------------------------------
    if intent == "faq":
        kb = search_kb(user_message, top_k=3)

        answer = "Here’s what I found in the support knowledge base:\n"
        answer += "\n".join([f"- {item['text']}" for item in kb["results"]])

        append_history(session_id, "assistant", answer)
        return {"route": "faq", "answer": answer, "kb": kb}

    # If an order id is present, do the operational lookup once and reuse it.
    order_info = None
    if order_id:
        order_info = lookup_order(order_id)
        if order_info.get("found"):
            context["order"] = order_info["order"]

    # ------------------------------------------------------------
    # Route B: Order lookup
    # ------------------------------------------------------------
    if intent == "order_lookup":
        if not order_info or not order_info.get("found"):
            answer = f"I couldn't find order {order_id or '(missing order id)'}."
            append_history(session_id, "assistant", answer)
            return {"route": "order_lookup", "answer": answer}

        order = order_info["order"]

        # Chain a KB lookup to enrich the response with support guidance.
        kb = search_kb("shipping updates delivered processing status", top_k=1)

        answer = (
            f"Order {order['order_id']} for {order['item']} is currently **{order['status']}**.\n"
            f"Support note: {kb['results'][0]['text']}"
        )

        append_history(session_id, "assistant", answer)
        return {"route": "order_lookup", "answer": answer, "order": order}

    # ------------------------------------------------------------
    # Route C: Refund / cancellation
    # ------------------------------------------------------------
    if intent in {"refund_request", "cancel_request"}:
        if not order_info or not order_info.get("found"):
            answer = "I need a valid order id to continue with that request."
            append_history(session_id, "assistant", answer)
            return {"route": intent, "answer": answer}

        # Pull policy context so the action is grounded in KB guidance.
        kb = search_kb(user_message, top_k=2)

        # Approval gate: pause the workflow and store the pending action in Redis.
        if needs_approval(intent, context) and not approved:
            state = load_session(session_id)
            state["pending"] = {
                "intent": intent,
                "order_id": order_id,
                "original_message": user_message
            }
            save_session(session_id, state)

            answer = (
                f"This action needs approval before I continue: **{intent}** for order {order_id}.\n"
                f"Reply with `APPROVE {order_id}` to continue."
            )

            append_history(session_id, "assistant", answer)
            return {
                "route": intent,
                "status": "approval_required",
                "answer": answer,
                "policy_context": kb["results"]
            }

        # If approval has already been granted, execute the action.
        if intent == "refund_request":
            refund = create_refund_request(order_id, reason=user_message)

            # Chain the async tool: queue a confirmation email after creating the refund.
            email_job = send_followup_email(
                to=order_info["order"]["customer_email"],
                subject=f"Refund request received for order {order_id}",
                body=f"We received your refund request {refund['request_id']}."
            )

            answer = (
                f"Refund request **{refund['request_id']}** has been submitted for order {order_id}. "
                f"A confirmation email was queued with job id **{email_job['job_id']}**."
            )

            append_history(session_id, "assistant", answer)
            return {
                "route": "refund_request",
                "refund": refund,
                "email_job": email_job,
                "answer": answer
            }

        # Simulated cancellation branch
        answer = f"Cancellation request for order {order_id} has been submitted for manual review."
        append_history(session_id, "assistant", answer)
        return {"route": "cancel_request", "answer": answer}

    # ------------------------------------------------------------
    # Route D: Async follow-up
    # ------------------------------------------------------------
    if intent == "async_followup":
        if order_info and order_info.get("found"):
            to = order_info["order"]["customer_email"]
            subject = f"Follow-up for order {order_id}"
        else:
            to = "customer@example.com"
            subject = "Support follow-up"

        job = send_followup_email(
            to,
            subject,
            f"Follow-up requested by customer: {user_message}"
        )

        answer = f"I queued the follow-up. Job id: **{job['job_id']}**."
        append_history(session_id, "assistant", answer)
        return {"route": "async_followup", "job": job, "answer": answer}

    # Fallback path (should rarely happen in this notebook)
    answer = "I couldn't determine the right route, so I treated this as a knowledge-base question."
    append_history(session_id, "assistant", answer)
    return {"route": "fallback", "answer": answer}


## 10) Resuming after approval

The approval gate above pauses a sensitive workflow and stores a **pending action** in Redis memory.

This helper simulates the next turn where a human approves the action:
- read the pending action from Redis
- verify the approval text
- resume the original request with `approved=True`


In [11]:

def continue_after_approval(session_id, approval_text):
    """
    Resume a previously paused sensitive action after human approval.
    """
    state = load_session(session_id)
    pending = state.get("pending")

    if not pending:
        return {
            "status": "no_pending_action",
            "answer": "There is no pending action waiting for approval."
        }

    order_id = pending["order_id"]

    if f"approve {order_id}".lower() not in approval_text.lower():
        return {
            "status": "not_approved",
            "answer": f"Approval text must include APPROVE {order_id}."
        }

    # Clear the pending action before resuming.
    state["pending"] = None
    save_session(session_id, state)

    return support_agent(session_id, pending["original_message"], approved=True)


## 11) Demo scenarios

### Demo 1 — FAQ / long-term recall
Ask a policy question and retrieve relevant KB chunks.

### Demo 2 — Order lookup
Look up an order and return its status using the operational tool.

### Demo 3 — Refund request with approval gate
Ask for a refund, trigger the approval gate, then resume after approval.

### Demo 4 — Async job status
The refund route queues a confirmation email.  
After a short delay, we call `check_job(job_id)` to show the async worker completed it.


In [12]:

# Start the background worker so queued email jobs can be processed.
worker_thread = run_worker_in_background()

session_id = "demo-session-1"

print("=== Demo 1: FAQ / long-term recall ===")
res1 = support_agent(session_id, "What is the refund policy for damaged items?")
print(res1["answer"])

print("\n=== Demo 2: Order lookup ===")
res2 = support_agent(session_id, "Where is order 1002?")
print(res2["answer"])

print("\n=== Demo 3: Refund request (approval gate) ===")
res3 = support_agent(session_id, "Refund order 1003 because the item arrived broken.")
print(res3["answer"])

print("\n=== Demo 3b: Approve pending action ===")
res4 = continue_after_approval(session_id, "APPROVE 1003")
print(res4["answer"])

# Give the worker a moment to process the queued email job.
time.sleep(1.0)

job_id = res4["email_job"]["job_id"]
print("\n=== Demo 4: Check async job ===")
print(check_job(job_id))


=== Demo 1: FAQ / long-term recall ===
I need a valid order id to continue with that request.

=== Demo 2: Order lookup ===
Order 1002 for Gaming Mouse is currently **processing**.
Support note: Standard shipping updates appear after carrier scan. Delivered orders with missing items should be escalated to support.

=== Demo 3: Refund request (approval gate) ===
This action needs approval before I continue: **refund_request** for order 1003.
Reply with `APPROVE 1003` to continue.

=== Demo 3b: Approve pending action ===
Refund request **rr_356be76e** has been submitted for order 1003. A confirmation email was queued with job id **job_f8ddfcda**.


/tmp/ipykernel_4101/4111481038.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat()
/tmp/ipykernel_4101/4111481038.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat()
/tmp/ipykernel_4101/4111481038.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat()



=== Demo 4: Check async job ===
{'job_id': 'job_f8ddfcda', 'status': 'queued', 'stream_id': '1782217017862-0'}


## 12) Inspect traces and memory

### Trace log
Shows the execution timeline:
- route decisions
- tool calls
- cache hits
- retry outcomes
- async worker completion

### Session memory
Shows what is stored in Redis short-term memory for the support session.


In [13]:
pd.DataFrame(TRACE_LOG).tail(30)

,ts,event,session_id,text,tool,order_id,query,fn,attempt,reason,to,subject
0,2026-06-23T12:16:57.814794,user_message,demo-session-1,What is the refund policy for damaged items?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-23T12:16:57.815391,user_message,demo-session-1,Where is order 1002?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-23T12:16:57.815419,tool_call,NaN,NaN,lookup_order,1002,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-23T12:16:57.815916,tool_call,NaN,NaN,search_kb,NaN,shipping updates delivered processing status,NaN,NaN,NaN,NaN,NaN
4,2026-06-23T12:16:57.840603,call_success,NaN,NaN,NaN,NaN,NaN,search,1.0,NaN,NaN,NaN
5,2026-06-23T12:16:57.841088,user_message,demo-session-1,Refund order 1003 because the item arrived bro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-06-23T12:16:57.841114,tool_call,NaN,NaN,lookup_order,1003,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-23T12:16:57.841213,tool_call,NaN,NaN,search_kb,NaN,Refund order 1003 because the item arrived bro...,NaN,NaN,NaN,NaN,NaN
8,2026-06-23T12:16:57.861765,call_success,NaN,NaN,NaN,NaN,NaN,search,1.0,NaN,NaN,NaN
9,2026-06-23T12:16:57.862388,user_message,demo-session-1,Refund order 1003 because the item arrived bro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
load_session("demo-session-1")

{'session_id': 'demo-session-1',
 'history': [{'role': 'user',
   'content': 'What is the refund policy for damaged items?',
   'ts': '2026-06-23T12:16:57.814730'},
  {'role': 'assistant',
   'content': 'I need a valid order id to continue with that request.',
   'ts': '2026-06-23T12:16:57.815075'},
  {'role': 'user',
   'content': 'Where is order 1002?',
   'ts': '2026-06-23T12:16:57.815365'},
  {'role': 'assistant',
   'content': 'Order 1002 for Gaming Mouse is currently **processing**.\nSupport note: Standard shipping updates appear after carrier scan. Delivered orders with missing items should be escalated to support.',
   'ts': '2026-06-23T12:16:57.840655'},
  {'role': 'user',
   'content': 'Refund order 1003 because the item arrived broken.',
   'ts': '2026-06-23T12:16:57.841053'},
  {'role': 'assistant',
   'content': 'This action needs approval before I continue: **refund_request** for order 1003.\nReply with `APPROVE 1003` to continue.',
   'ts': '2026-06-23T12:16:57.861885'},